#  Soccer Transfer Value Estimator (Interactive, Guided Version)

Loads models for Forwards, Midfielders, and Defenders, then launches an interactive app where users can input player stats with helpful guidance and realistic limits.oo

In [1]:
!pip install gradio scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 5.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import joblib
import gradio as gr
from sklearn.pipeline import Pipeline

In [3]:
# uploading model pkl files
from google.colab import files
uploaded = files.upload()

pipeline_offense = joblib.load("pipeline_offense.pkl")
pipeline_midfield = joblib.load("pipeline_midfield.pkl")
pipeline_defense = joblib.load("pipeline_defense.pkl")

Saving pipeline_defense.pkl to pipeline_defense.pkl
Saving pipeline_midfield.pkl to pipeline_midfield.pkl
Saving pipeline_offense.pkl to pipeline_offense.pkl


In [5]:
def predict_transfer_value(position, age_group, ga=0, expected_contribution=0.0, shots=0,
                           assists=0, xag=0.0, kp=0, prgp=0, prgc=0,
                           tackles=0, blocks=0, pass_blocks=0):
    if position == "Forward":
        input_df = pd.DataFrame([{
            "G+A": ga,
            "Expected_Contribution": expected_contribution,
            "Sh_Standard": shots,
            "Position": position,
            "Age_Group": age_group
        }])
        prediction = pipeline_offense.predict(input_df)[0]

    elif position == "Midfielder":
        input_df = pd.DataFrame([{
            "Ast": assists,
            "xAG": xag,
            "KP": kp,
            "PrgP": prgp,
            "PrgC_Carries": prgc,
            "Position": position,
            "Age_Group": age_group
        }])
        prediction = pipeline_midfield.predict(input_df)[0]

    elif position == "Defender":
        input_df = pd.DataFrame([{
            "Att 3rd_Tackles": tackles,
            "Blocks_Blocks": blocks,
            "Pass_Blocks": pass_blocks,
            "Position": position,
            "Age_Group": age_group
        }])
        prediction = pipeline_defense.predict(input_df)[0]
    else:
        return "Invalid position selected."

    return f"Estimated Market Value: €{prediction:,.0f}"

In [6]:
with gr.Blocks() as demo:
    gr.Markdown("### Soccer Player Transfer Value Estimator")

    with gr.Accordion("Stat Glossary", open=False):
        gr.Markdown("""
- **G+A**: Goals + Assists combined
- **Expected Contribution**: Sum of Expected Goals and Expected Assists (xG + xAG)
- **Shots**: Total number of shots attempted
- **xAG**: Expected Assists based on shot quality
- **KP**: Key Passes leading directly to a shot
- **PrgP**: Progressive passes toward opponent's goal
- **PrgC**: Progressive carries into attacking zones
- **Blocks / Pass Blocks**: Defensive actions preventing passes or shots
        """)

    position = gr.Dropdown(["Forward", "Midfielder", "Defender"], label="Position")
    age_group = gr.Dropdown(["Young", "Prime", "Veteran"], label="Age Group")

    gr.Markdown("### Enter Player Stats (only use those relevant for selected position)")

    ga = gr.Number(label="Goals + Assists", maximum=44, info="Normal: 5–20, Elite: >30")
    expected_contribution = gr.Number(label="Expected Contribution (xG + xAG)", maximum=37.0, info="Solid: 10–20, Elite: >30")
    shots = gr.Number(label="Shots", maximum=141, info="Most forwards have 30–100 shots")

    assists = gr.Number(label="Assists", maximum=13, info="5–10 is good, 13+ elite")
    xag = gr.Number(label="xAG", maximum=11.8, info="Expected assists from passes; 5+ is high")
    kp = gr.Number(label="Key Passes", maximum=60, info="High = 40+, Moderate = 20")
    prgp = gr.Number(label="Progressive Passes", maximum=180, info="Progresses ball downfield")
    prgc = gr.Number(label="Progressive Carries", maximum=120, info="Carries into final third")

    tackles = gr.Number(label="Tackles in Attacking Third", maximum=30, info="10–25 is typical for active defenders")
    blocks = gr.Number(label="Blocks", maximum=80, info="High block volume = 60+")
    pass_blocks = gr.Number(label="Pass Blocks", maximum=45, info="Defensive blocks of passes")

    predict_btn = gr.Button("Predict Value")
    output = gr.Textbox(label="Estimated Transfer Value (€)")

    predict_btn.click(predict_transfer_value,
        inputs=[position, age_group, ga, expected_contribution, shots,
                assists, xag, kp, prgp, prgc,
                tackles, blocks, pass_blocks],
        outputs=output)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6474e8d0812cb718b5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!pip install qrcode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 1.8 MB/s eta 0:00:00


In [ ]:
import qrcode
from PIL import Image

url = "https://0d142069da6c048deb.gradio.live"

qr = qrcode.make(url)
qr.save("transfer_value_qr.png")

Image.open("transfer_value_qr.png").show()

In [ ]:
from google.colab import files
files.download("transfer_value_qr.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>